In [5]:
import numpy as np
import matplotlib.pyplot as plt

# 1. 构造转移矩阵 (增加 N 以获得更多有效特征值)
r = 1.5437
N = 2000
edges = np.linspace(-1, 1, N + 1)
centers = (edges[:-1] + edges[1:]) / 2
P = np.zeros((N, N))

for j in range(N):
    x_next = 1 - r * (centers[j]**2)
    if -1 <= x_next <= 1:
        i = np.searchsorted(edges, x_next) - 1
        if 0 <= i < N:
            P[i, j] = 1.0

# 2. 提取特征值
eigvals = np.linalg.eigvals(P)
# 取模长，并排序
mags = np.sort(np.abs(eigvals))

# 3. 鲁棒性过滤：排除最大的 1.0 和由于数值误差产生的接近 0 的值
# 这里我们采用分位数过滤，确保一定有数据剩下
threshold_low = 0.091
threshold_high = 0.999
filtered_mags = mags[(mags > threshold_low) & (mags < threshold_high)]

if len(filtered_mags) < 2:
    print(f"警告：有效特征值太少（仅 {len(filtered_mags)} 个），请尝试增大 N 或放宽过滤阈值。")
else:
    # 4. 谱展开 (Unfolding)
    # 对于 1D 映射，最简单有效的展开是直接计算归一化间距
    unfolded = np.sort(filtered_mags)
    spacings = np.diff(unfolded)
    
    # 过滤掉由于离散化导致的完全相同的特征值 (即间距为0的情况)
    spacings = spacings[spacings > 1e-10]
    
    if len(spacings) > 0:
        s = spacings / np.mean(spacings)

        # 5. 绘图
        plt.figure(figsize=(10, 6))

        # 实验数据直方图
        plt.hist(s, bins=50, density=True, alpha=0.6, color='#5DADE2', label='Quadratic Map Spectrum')

        # GUE 理论曲线 (黎曼零点/强混沌)
        x_plot = np.linspace(0, 4, 100)
        gue = (32 / (np.pi**2)) * (x_plot**2) * np.exp(-(4 / np.pi) * (x_plot**2))
        plt.plot(x_plot, gue, 'r-', lw=2.5, label='GUE (Riemann/Chaos Prediction)')

        # Poisson 理论曲线 (规则系统)
        poisson = np.exp(-x_plot)
        plt.plot(x_plot, poisson, 'g--', lw=2, label='Poisson (Regular)')

        plt.title(f"Level Spacing Distribution (N={N}, r={r})", fontsize=14)
        plt.xlabel("Normalized Spacing s", fontsize=12)
        plt.ylabel("P(s)", fontsize=12)
        plt.xlim(0, 3.5)
        plt.legend()
        plt.grid(alpha=0.2)
        plt.show()
        
        # 额外：看看复平面上的分布，确认极点位置
        plt.figure(figsize=(5, 5))
        plt.scatter(eigvals.real, eigvals.imag, s=1, alpha=0.5, c='purple')
        plt.title("Eigenvalue Distribution on Complex Plane")
        plt.xlabel("Re")
        plt.ylabel("Im")
        plt.show()
    else:
        print("无法计算间距，请检查特征值分布。")

警告：有效特征值太少（仅 0 个），请尝试增大 N 或放宽过滤阈值。
